# Sommeil EOG — CNN NPU (reprise Kaggle)

Reprend l'entrainement CNN 1D depuis le PC (8 epochs deja faites).

## Fichier checkpoint pour Kaggle (IMPORTANT)

Uploadez **`sleep_model_cnn_best_weights.npz`** (~1,7 Mo).

Genere sur PC : `python scripts/export_cnn_weights_npz.py`

## + Add data

1. Dataset **corpus** : sleep_edf_corpus.npz + 2 json
2. Dataset **poids** : `sleep_model_cnn_best_weights.npz` (PAS le .keras ni .h5)

Puis **Run All**. GPU P100.

In [ ]:
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score
from sklearn.utils import class_weight
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (
    BatchNormalization, Conv1D, Dense, Dropout,
    GlobalAveragePooling1D, MaxPooling1D,
)
from tensorflow.keras.models import Sequential

SEED = 42
N_SAMPLES = 3000
EPOCHS_TOTAL = 30
INITIAL_EPOCH = 7   # epochs deja faites sur PC (ajuster si besoin)
BATCH_SIZE = 64
PATIENCE = 7
STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]
CORPUS_FILES = ("sleep_edf_corpus.npz", "sleep_edf_corpus_meta.json", "subject_split.json")
WEIGHTS_NPZ = "sleep_model_cnn_best_weights.npz"

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def list_input_files():
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    return sorted(p for p in root.glob("**/*") if p.is_file())


def find_file(name):
    hits = [p for p in list_input_files() if p.name == name]
    if hits:
        return hits[0]
    found = [p.name for p in list_input_files()]
    msg = (
        f"{name} introuvable sous /kaggle/input.\n\n"
        f"Fichiers vus ({len(found)}) :\n"
        + "\n".join(f"  - {n}" for n in found[:30])
        + ("\n  ..." if len(found) > 30 else "")
        + "\n\nFix : + Add data → attachez 2 datasets (corpus + checkpoint) "
        "OU uploadez les 4 fichiers dans une seule version."
    )
    raise FileNotFoundError(msg)


def corpus_dir_from_npz(npz_path):
    d = npz_path.parent
    if all((d / f).exists() for f in CORPUS_FILES):
        return d
    raise FileNotFoundError("Dataset corpus incomplet (3 fichiers requis).")


print("=== Fichiers dans /kaggle/input ===")
for p in list_input_files():
    print(f"  {p.relative_to('/kaggle/input')}")

NPZ_PATH = find_file("sleep_edf_corpus.npz")
INPUT_DIR = corpus_dir_from_npz(NPZ_PATH)

tf.random.set_seed(SEED)
np.random.seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("\nCorpus       :", INPUT_DIR)
print("GPU          :", tf.config.list_physical_devices("GPU"))
print(f"Reprise      : epoch {INITIAL_EPOCH + 1} -> {EPOCHS_TOTAL}")

In [ ]:
def masks_from_manifest(subject_idx, subject_names, manifest):
    name_to_id = {name: i for i, name in enumerate(subject_names)}
    def _mask(names):
        ids = {name_to_id[n] for n in names if n in name_to_id}
        return np.isin(subject_idx, list(ids))
    return _mask(manifest["train_subjects"]), _mask(manifest["val_subjects"]), _mask(manifest["test_subjects"])


class F1MacroCallback(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, patience=PATIENCE):
        super().__init__()
        self.X_val, self.y_val = X_val, y_val
        self.patience, self.best_f1, self.wait = patience, -1.0, 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
        macro = float(f1_score(self.y_val, y_pred, average="macro", zero_division=0))
        logs["val_f1_macro"] = macro
        print(f"  -> val_f1_macro = {macro:.4f}")
        if macro > self.best_f1 + 1e-4:
            self.best_f1, self.wait = macro, 0
        else:
            self.wait += 1
        if self.wait >= self.patience:
            self.model.stop_training = True


data = np.load(NPZ_PATH)
X, y = data["X"].astype(np.float32), data["y"].astype(np.int32)
subject_idx = data["subject_idx"].astype(np.int32)
with open(INPUT_DIR / "sleep_edf_corpus_meta.json", encoding="utf-8") as f:
    subject_names = json.load(f)["subject_names"]
with open(INPUT_DIR / "subject_split.json", encoding="utf-8") as f:
    manifest = json.load(f)
train_mask, val_mask, _ = masks_from_manifest(subject_idx, subject_names, manifest)
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
del data

class_weights = dict(enumerate(class_weight.compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)))
print(f"Train {len(y_train):,} epoques · Val {len(y_val):,} epoques")


def build_cnn_npu_model(input_shape=(N_SAMPLES, 1), num_classes=5):
    """Identique a src/architecture.py — build_cnn_npu_model."""
    model = Sequential([
        Conv1D(64, kernel_size=11, strides=1, activation="relu", padding="same", input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=4),
        Dropout(0.2),
        Conv1D(128, kernel_size=7, strides=1, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=4),
        Dropout(0.3),
        Conv1D(256, kernel_size=5, strides=1, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=4),
        Dropout(0.3),
        Conv1D(256, kernel_size=3, strides=1, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        GlobalAveragePooling1D(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def load_weights_from_npz(model, npz_path):
    data = np.load(npz_path)
    loaded, missing = 0, []
    for layer in model.layers:
        keys = sorted(k for k in data.files if k.startswith(layer.name + "__"))
        if not keys:
            if layer.weights:
                missing.append(layer.name)
            continue
        layer.set_weights([data[k] for k in keys])
        loaded += 1
    print(f"Poids NPZ : {loaded} couches chargees depuis {npz_path}")
    if missing:
        print("Couches sans poids (normal si Dropout/Pool) :", missing[:5])


def load_model_for_training(input_shape=(N_SAMPLES, 1)):
    model = build_cnn_npu_model(input_shape=input_shape)
    hits = [p for p in list_input_files() if p.name == WEIGHTS_NPZ]
    if hits:
        load_weights_from_npz(model, str(hits[0]))
        return model, INITIAL_EPOCH
    print("ATTENTION :", WEIGHTS_NPZ, "absent -> entrainement depuis epoch 1")
    return model, 0

In [ ]:
model, start_epoch = load_model_for_training(input_shape=(X_train.shape[1], 1))
model.summary()

best_path = WORK_DIR / "sleep_model_cnn_best.keras"
callbacks = [
    F1MacroCallback(X_val, y_val),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(best_path), monitor="val_f1_macro", mode="max", save_best_only=True, verbose=1),
]

print(f"Entrainement epoch {start_epoch + 1} -> {EPOCHS_TOTAL}")
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    initial_epoch=start_epoch,
    epochs=EPOCHS_TOTAL,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    shuffle=True,
    verbose=2,
)

if best_path.exists():
    model = tf.keras.models.load_model(best_path)
model.save(WORK_DIR / "sleep_model_cnn.keras")
print("Termine — telechargez sleep_model_cnn.keras et sleep_model_cnn_best.keras")